#**Proyecto Integrador - Minería de Datos 1**
**05 - Conclusiones**


## Carga del dataset procesado

In [ ]:
import pandas as pd
import numpy as np

try:
    df = pd.read_csv('../data/processed/streaming_users_clean.csv')
except FileNotFoundError:
    df = pd.read_csv('streaming_users_clean.csv')

print(f'Dataset procesado: {df.shape[0]} filas x {df.shape[1]} columnas')

## Respuesta a las preguntas de análisis


In [ ]:
# Distribución de edad
media_age = df['age'].mean()
mediana_age = df['age'].median()
std_age = df['age'].std()
skew_age = df['age'].skew()

print('=== Distribución de edad ===')
print(f'Media:    {media_age:.2f} años')
print(f'Mediana:  {mediana_age:.2f} años')
print(f'Desvío:   {std_age:.2f} años')
print(f'Asimetría:{skew_age:.4f}')

**Pregunta 1 — ¿Cómo se distribuye la edad de los usuarios?**

La distribución de edad presenta media = 33.46 años, mediana = 33.00 años
y asimetría = 0.15, prácticamente simétrica. El desvío estándar es de 11.79 años,
con usuarios distribuidos entre 0 y 100 años tras la limpieza.

La cercanía entre media y mediana indica que no hay distorsión por
valores extremos. La distribución uniforme a lo largo del rango sugiere que la plataforma
no está concentrada en un nicho etario particular.

Como conclusión, no existe una franja etaria dominante en el dataset.

In [ ]:
# Distribución del tiempo de visualización
media_wt = df['monthly_watch_time_mins'].mean()
mediana_wt = df['monthly_watch_time_mins'].median()
skew_wt = df['monthly_watch_time_mins'].skew()

print('=== Distribución del tiempo de visualización mensual ===')
print(f'Media:    {media_wt:.2f} min')
print(f'Mediana:  {mediana_wt:.2f} min')
print(f'Asimetría:{skew_wt:.4f}')

**Pregunta 2 — ¿Cómo se distribuye el tiempo de visualización mensual?**

La distribución presenta asimetría positiva (skew = 1.25), con media
(786.98 min) superior a la mediana (758.50 min). La mayoría de los usuarios se concentra
en valores bajos-moderados.

La asimetría positiva indica la presencia de usuarios con consumo
notablemente superior al típico. La brecha media-mediana confirma que
la media está siendo arrastrada hacia arriba por esos valores altos, siendo la mediana
la medida más representativa del usuario típico.

Entonces podemos inferir que usuario típico de la plataforma consume aproximadamente 758 minutos
mensuales (~12.6 horas).

In [ ]:
#Tiempo de visualización por plan
stats_plan = df.groupby('subscription_plan')['monthly_watch_time_mins'].agg(
    ['mean', 'median']
).round(2)
stats_plan.columns = ['Media (min)', 'Mediana (min)']
print('=== Tiempo de visualización por plan ===')
print(stats_plan)

**Pregunta 3 — ¿El tiempo de visualización varía según el plan de suscripción?**

Las medias por plan son: Básico = 592.73 min, Estándar = 861.42 min,
Premium = 1091.15 min. Los boxplots del EDA muestran que las cajas (50% central)
apenas se superponen entre planes.

La diferencia entre Básico y Premium es de aproximadamente 498 minutos
mensuales (~8.3 horas). La separación entre grupos es robusta: no es producto de pocos
valores extremos sino de una diferencia estructural en el comportamiento de consumo.

En conclusión, el plan de suscripción está fuertemente asociado al nivel de consumo
mensual. Sin embargo, con los datos disponibles no es posible determinar la dirección
de la causalidad: no puede afirmarse si el plan determina el consumo o si los usuarios
que más consumen tienden a elegir planes superiores.

In [ ]:
# Correlacion edad vs tickets
r = df['age'].corr(df['customer_support_tickets'])
print("=== Correlacion age vs tickets ===")
print(f"Pearson r = {r:.4f}")

**Pregunta 4 — ¿Existe relación entre la edad y la cantidad de tickets de soporte?**

El coeficiente de correlación de Pearson entre *age* y
*customer_support_tickets* es r = 0.0054, prácticamente igual a cero. El scatter plot
del EDA confirma visualmente la ausencia de tendencia.

Un r ≈ 0 descarta la existencia de relación lineal entre estas dos
variables. Esto refuta hipótesis como: "los usuarios mayores generan más
tickets por menor familiaridad tecnológica" o "los usuarios jóvenes generan más tickets
por uso más intensivo". Los datos no respaldan ninguna de esas hipótesis.

Obtenemos como resultado que la edad no es un predictor del uso del soporte técnico en este dataset.
La cantidad de tickets de soporte parece ser independiente del perfil etario del usuario.

In [ ]:
# Consumo por pais y plan
pivot = df.pivot_table(
    values='monthly_watch_time_mins',
    index='country',
    columns='subscription_plan',
    aggfunc='mean'
)[['Básico', 'Estándar', 'Premium']].round(1)

print("=== Promedio de minutos por pais y plan ===")
print(pivot)

print("")
print("=== Rango de variacion entre paises por plan ===")
for plan in ['Básico', 'Estándar', 'Premium']:
    rango = pivot[plan].max() - pivot[plan].min()
    print(f"{plan}: {rango:.1f} min de diferencia entre paises")


**Pregunta 5 — ¿El perfil de consumo por plan es consistente entre países?**

El heatmap del EDA muestra que la jerarquía Premium > Estándar > Básico
se mantiene sin excepciones en los 8 países del dataset. La variación entre países
dentro de cada plan es menor al 10% (rango < 100 min en todos los casos).

El plan de suscripción es el factor dominante en el nivel de consumo,
por encima de la ubicación geográfica. No existe un país donde el patrón de consumo
por plan sea atípico o inverso.

En conclusión, el comportamiento de consumo según plan es uniforme a nivel regional.
Esto sugiere que las diferencias culturales o de infraestructura entre países no
generan variaciones significativas en el tiempo de visualización dentro de cada
segmento de plan.

## Limitaciones

El alcance de las conclusiones se encuentra condicionado por la información disponible
y por las decisiones documentadas durante el proceso.

1. El dataset no incluye variables de contexto como tipo de
   dispositivo, hora de uso, o historial de contenido consumido, que podrían enriquecer
   el análisis del perfil de usuario.

2. Las asociaciones observadas (especialmente entre plan
   y consumo) no permiten establecer relaciones causales con los datos disponibles.
   Por ejemplo para determinar si el plan determina el consumo o viceversa.

3. La baja correlación entre las tres
   variables numéricas limitó la capacidad de reducción de dimensionalidad del PCA,
   que no logró comprimir la información en menos componentes sin pérdida significativa.

4. El proceso de limpieza eliminó el 13.4% de los registros
   originales. Si bien cada decisión fue justificada con evidencia, la eliminación de
   registros introduce un sesgo de selección potencial: los registros eliminados podrían
   no ser aleatorios respecto al comportamiento de los usuarios.

## Mejoras futuras

Una mejora futura podría consistir en incorporar variables adicionales que permitan
ampliar el alcance del análisis, como el historial de contenido consumido por género
o el tipo de dispositivo utilizado.

Otra mejora posible sería explorar técnicas de clustering (como K-Means) para
identificar segmentos de usuarios con comportamiento similar, complementando los
hallazgos del EDA con una agrupación basada en múltiples variables simultáneamente.

Finalmente, la incorporación de datos longitudinales (múltiples registros por usuario
a lo largo del tiempo) permitiría analizar la evolución del consumo y abordar preguntas
sobre retención y churn de usuarios.

## Síntesis final

Los hallazgos más relevantes son:

- El *plan de suscripción* es el factor más fuertemente asociado al nivel de consumo
  mensual, con una diferencia de casi 500 minutos entre el plan Básico y el Premium,
  consistente en todos los países del dataset.

- La *edad* no mostró relación con ninguna otra variable numérica del dataset
  (r ≈ 0 con tickets de soporte), lo que indica que el perfil etario no determina
  el comportamiento de uso en esta plataforma.
- El *PCA* confirmó la independencia entre las variables numéricas: la varianza
  se distribuye uniformemente entre los tres componentes.

